# Linear-regression baselines for the Stage-1 reconstruction

**Question.** The GAT+BiLSTM reconstructs the tension along a mooring line from the motion of its
stations. Is a graph network with a bidirectional LSTM needed, or would linear regression do?

**How this notebook answers it.** It fits five linear models and scores them on exactly the
problem Stage 1 solves:
- the same inputs;
- the same training windows;
- the same test windows;
- the same metric code.

Every number is then directly comparable with the published Stage-1 model (v5, epoch 32).

| model | inputs for station *i* at time *t* | inputs |
|---|---|---|
| **LR0_static** | none fitted: the station's static tension `tension0`, held constant | 0 |
| **LR1_pointwise** | the station's own inputs at *t*, exactly what the GAT receives per station (12 static, 22 dynamic) | 34 |
| **LR2_line** | LR1 plus the 9 motion channels of the fairlead, of the two neighbouring stations and their mean along the line | 70 |
| **LR3_lagged** | LR2 plus the station's and the fairlead's 9 motion channels at lags of ±0.1…±5 s: a linear filter over the window, non-causal like the BiLSTM | 286 |
| **LR4_lagged40** | LR3 plus the same channels at every 1 s out to ±40 s: a linear filter whose memory spans most of the 100 s window | 1,618 |

**Removed before fitting.** Two static inputs are exact linear functions of others, so they carry no
information a linear model could use, and are left out of every fit:
- `is_intermediate` = 1 − `is_anchor` − `is_fairlead`;
- `submerged_weight` is a fixed multiple of `nodal_mass`.

Kept in, float32 rounding makes each pair differ by about 1e-7, and least squares fits that noise
with huge cancelling weights.

**The ladder separates information from linearity.**
- LR1 is the plain linear regression: one station at one instant.
- LR2 and LR3 give a linear model the information the neural network uses: the whole line, and
  the time history inside the 100 s window.
- LR4 extends that history to ±40 s, longer than any memory of the system: the longest wave
  period is about 18 s, and a transverse wave crosses the line in 4–18 s. Its columns are LR3's
  plus the longer lags, so the validation error can be read as a function of the memory span
  (section 6), from one fit.
- If LR3 and LR4 still fall short, the gap comes from the nonlinearity of the map, not from
  missing information.

**Fitting.** Ordinary least squares, the exact minimiser of the mean squared error. The fit is
streamed as a QR factorisation over standardised inputs (see `linear_regression_lib.py`). LR3 and
LR4 may add a small ridge term, chosen on the validation split.

**What is reused, unchanged.** The graph builder, resampling, per-window input features, split
utilities, evaluation-mask filter and per-window metric function are copied verbatim from the
Stage-1 notebook into `stage1_verbatim.py` (generated by `_build_linear_regression_nb.py`). Gates
below check that the split, the inputs, the targets and the metric aggregation match Stage 1.

**Running it.**
- `LR_RUN_MODE=smoke` (the default) runs everything on six simulations in a few minutes.
- `LR_RUN_MODE=full` runs the full data: tens of CPU-core hours, most of it LR4's fit. On a
  48-core cluster node `run_LR_baseline.slurm` runs the smoke check and then the full run.
- Memory: each worker process peaks at about 0.9 GB. The number of workers is set from the
  cores and from the memory (the Slurm allocation, or what is free when the run starts);
  `LR_WORKERS` overrides it. On a 16 GB laptop that allows about three workers, so the full
  run takes several hours there.
- Headless: `LR_RUN_MODE=full python -m nbconvert --to notebook --execute Linear_Regression.ipynb`.

In [ ]:
import os
import sys
import time
import json
import random
import hashlib
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))


class _Tee:
    """Copies everything printed to the file LR_PROGRESS_LOG, so a headless run (nbconvert,
    e.g. the cluster job run_LR_baseline.slurm) can be followed while it runs."""
    _lr_tee = True

    def __init__(self, stream, path):
        self._stream = stream
        self._file = open(path, "a", encoding="utf-8", buffering=1)

    def write(self, s):
        self._file.write(s)
        return self._stream.write(s)

    def flush(self):
        self._file.flush()
        self._stream.flush()

    def __getattr__(self, name):
        return getattr(self._stream, name)


if os.environ.get("LR_PROGRESS_LOG") and not getattr(sys.stdout, "_lr_tee", False):
    sys.stdout = _Tee(sys.stdout, os.environ["LR_PROGRESS_LOG"])

CACHE_DIR = Path(os.environ.get("MOORING_CACHE_DIR", r"C:\Users\thano\Desktop\data\cache_npy"))
RUN_MODE = os.environ.get("LR_RUN_MODE", "smoke")            # "smoke" | "full"
assert RUN_MODE in ("smoke", "full"), RUN_MODE

# Worker processes, sized by memory as well as by cores. One worker peaks at about 0.9 GB
# (measured, pass A, one thread); ten of them froze a 16 GB laptop. Under Slurm the job's
# allocation is used, otherwise the memory that is free when the run starts.
WORKER_GB = 1.2


def _cpus():
    if os.environ.get("SLURM_CPUS_PER_TASK"):
        return int(os.environ["SLURM_CPUS_PER_TASK"])
    if hasattr(os, "sched_getaffinity"):
        return len(os.sched_getaffinity(0))
    return os.cpu_count() or 2


def _memory_gb():
    if os.environ.get("SLURM_MEM_PER_NODE"):                  # Slurm gives MiB
        return int(os.environ["SLURM_MEM_PER_NODE"]) * 2**20 / 1e9
    if os.environ.get("SLURM_MEM_PER_CPU"):
        return int(os.environ["SLURM_MEM_PER_CPU"]) * _cpus() * 2**20 / 1e9
    import psutil
    return psutil.virtual_memory().available / 1e9


ON_SLURM = "SLURM_JOB_ID" in os.environ
CPUS, MEM_GB = _cpus(), _memory_gb()
N_WORKERS = int(os.environ.get("LR_WORKERS") or
                max(1, min(CPUS - 1, int((0.8 * MEM_GB - 2.0) / WORKER_GB))))
RSS_LIMIT_GB = float(os.environ.get("LR_RSS_LIMIT_GB") or 0.9 * MEM_GB)
MIN_AVAIL_GB = float(os.environ.get("LR_MIN_AVAIL_GB") or (0.0 if ON_SLURM else 1.5))
# On the cluster the Stage-1 reference files must be present: a gate that is skipped proves nothing.
REQUIRE_REFERENCES = os.environ.get("LR_REQUIRE_REFERENCES", "0") == "1"

# Stage-1 run configuration (split seed, budgets) and the Stage-1 reference outputs used by the
# gates and the comparison. The gates are skipped, with a message, if a reference is absent.
STAGE1_CONFIG = ROOT / "Results" / "checkpoints_P1_matched_v5" / "config.json"
STAGE1_EVAL = ROOT / "Results" / "SR_RESULTS" / "checkpoints_P2_eval_clean" / "stage1"
STAGE1_DUMPS = {"v5": ROOT / "Results" / "checkpoints_P1_matched_v5" / "test_timeseries_dump.npz",
                "snapfix": ROOT / "Results" / "eval_v5_snapfix" / "test_timeseries_dump.npz"}

OUT = ROOT / "Results" / "LR_baseline" / ("" if RUN_MODE == "full" else "_smoke")
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
MOMENT_WINDOWS, MOMENT_STRIDE = 5000, 5     # standardiser: 5,000 train windows, every 5th sample
LR3_WINDOW_FRACTION, LR3_STRIDE = 0.10, 5   # LR3: 10 % of the pool, every 5th sample
LR4_STRIDE, LR4_VAL_STRIDE = LR3_STRIDE, 5  # LR4: LR3's windows and rows; validation every 5th sample
LR4_BUDGET, LR4_CAP = 600e6, 64             # rows per QR update of the wide LR4 design (bytes, windows)
# LR4's tasks are larger: each returns a 1,620 x 1,620 factor, and merging ~1,200 of them in this
# process would take ~25 GB and tens of minutes (still several tasks per worker at 12).
LR4_CASES_PER_TASK = int(os.environ.get("LR4_CASES_PER_TASK") or (12 if RUN_MODE == "full" else 1))
# LR0-LR3 outputs of the first cluster run (job 872463): this run must reproduce them (GATE 8)
REF_872463 = ROOT / "Results" / "LR_ref_872463"
RIDGE_GRID = [0.0] + [10.0 ** k for k in range(-8, 0)]
SMOKE_CASES = {(1, 7), (4, 1), (4, 2), (6, 5), (10, 1), (11, 3)}
CASES_PER_TASK = int(os.environ.get("LR_CASES_PER_TASK") or (4 if RUN_MODE == "full" else 1))

print(f"mode {RUN_MODE} | {CPUS} CPUs, {MEM_GB:.0f} GB -> {N_WORKERS} worker processes | "
      f"memory guard: RSS <= {RSS_LIMIT_GB:.0f} GB, free >= {MIN_AVAIL_GB:.1f} GB | "
      f"{CASES_PER_TASK} simulation(s) per task")
print(f"cache {CACHE_DIR} | out {OUT}")

In [ ]:
import inspect
import numpy as np
import torch
import linear_regression_lib as L
import stage1_verbatim as S1

L.RSS_LIMIT_GB, L.MIN_AVAIL_GB = RSS_LIMIT_GB, MIN_AVAIL_GB
print("python", sys.version.split()[0], "| numpy", np.__version__, "| torch", torch.__version__,
      "| threads per process", torch.get_num_threads())
print("model inputs per rung:", {r: L.rung_cols(r) for r in L.RUNGS})
print(f"LR4 lags: {len(L.LAGS_LONG)} (LR3's {len(L.LAGS)}, then every 1 s to "
      f"+-{L.LONG_LAG_MAX * 0.1:.0f} s)")
print(inspect.getsource(L.design))

## 1. Data tables and the Stage-1 configuration

**The masks.**
- The training mask (`artifact_samples.csv`: shape rule, 50 kN ceiling, non-finite values) removes
  every window that covers a corrupt sample, in training and in testing.
- The evaluation mask (`artifact_samples_cat.csv`) adds the samples where a station exceeds the
  fairlead. It thins only the test windows.

**The snap catalogue** (`snap_catalogue_cat.csv`) lists the 902 snap events that the test dumps
are scored on.

The counts are asserted: they must equal the tables embedded in the Stage-1 notebook.

In [ ]:
T = L.load_tables(ROOT)
count = lambda d: (len(d), sum(len(v) for v in d.values()))  # noqa: E731
assert count(T["artifact"]) == (109, 1643), count(T["artifact"])
assert count(T["catenary"]) == (80, 7772), count(T["catenary"])
assert count(T["snap_times"]) == (182, 902), count(T["snap_times"])
L.set_tables(T["artifact"])

cfg = L.load_split_config(STAGE1_CONFIG)
assert (cfg.split_seed, cfg.window_len, cfg.snap_oversample_frac) == (42, 1000, 0.0)
assert list(cfg.node_counts) == list(L.LAYOUTS)
assert (list(cfg.train_lc_ids), list(cfg.test_extra_lc_ids)) == (list(L.DEV_SITES), [10, 11])
print({k: getattr(cfg, k) for k in L.SPLIT_FIELDS})

## 2. The split — rebuilt, and checked against Stage 1

**How it is rebuilt.**
- The Stage-1 split utilities run on header-only stand-ins, in Stage 1's dataset order
  (site → case → layout).
- Only the record length of each simulation is read; no time series is opened.

**GATE 1.** The rebuilt test draw must equal, window for window, the 50,000 windows Stage 1 was
scored on. After the evaluation mask it must equal the 49,513 windows the paper reports.

In [ ]:
t0 = time.time()
SPLIT = L.regenerate_split(CACHE_DIR, cfg, T["catenary"])
print(f"split rebuilt in {time.time() - t0:.0f} s: " +
      ", ".join(f"{k} {len(SPLIT[k]):,}" for k in ("train", "val", "test", "test_screened")))

def ref_keys(path):
    return {(int(r["lc_id"]), int(r["case_id"]), int(r["n_in"]), int(r["start_idx"]))
            for r in L.read_rows(path)}

for name, split_key, ref in (("published", "test", STAGE1_EVAL / "published" / "test_per_window_stats.csv"),
                             ("screened", "test_screened", STAGE1_EVAL / "test_per_window_stats.csv")):
    if ref.exists():
        mine, theirs = set(SPLIT[split_key]), ref_keys(ref)
        assert mine == theirs, f"GATE 1 FAILED ({name}): {len(mine ^ theirs)} windows differ"
        print(f"GATE 1 ({name}): the {len(mine):,} test windows equal Stage 1's exactly -- PASS")
    else:
        assert not REQUIRE_REFERENCES, f"GATE 1 ({name}): reference {ref} not found"
        print(f"GATE 1 ({name}): reference {ref} not found -- skipped")

def restrict(keys):
    return list(keys) if RUN_MODE == "full" else [k for k in keys if (k[0], k[1]) in SMOKE_CASES]

TRAIN, VAL, TEST = restrict(SPLIT["train"]), restrict(SPLIT["val"]), restrict(SPLIT["test"])
SCREENED = set(restrict(SPLIT["test_screened"]))
print(f"used here: train {len(TRAIN):,} | val {len(VAL):,} | test {len(TEST):,} "
      f"(screened {len(SCREENED):,})")

## 3. Inputs — checked against the Stage-1 dataset class

**The batched feature builder** computes the 22 dynamic inputs for a stack of windows at once.

**GATE 2.** Its output, and the target, must equal what Stage 1's own
`MooringSequenceDatasetPositionTension.__getitem__` returns for the same window. The check covers
every layout and all three splits.

In [ ]:
rng = random.Random(SEED)
gate_keys = []
for pool in (TRAIN, VAL, TEST):
    by_n = {}
    for k in pool:
        by_n.setdefault(k[2], []).append(k)
    for n_st, ks in sorted(by_n.items()):
        gate_keys += rng.sample(ks, min(3, len(ks)))
n_chk, dx, dy = L.check_features(CACHE_DIR, gate_keys)
assert dx == 0.0 and dy == 0.0, f"GATE 2 FAILED: max |dx| {dx}, max |dy| {dy}"
print(f"GATE 2: {n_chk} windows, inputs and targets identical to the Stage-1 class -- PASS")

## 4. Standardisation

**What.** Column means and standard deviations, from 5,000 random training windows (every 5th
sample). Near-constant columns keep a scale of 1, as in Stage 1's `FeatureStandardizer`.

**Effect.** Least-squares predictions do not depend on this scaling. It only keeps the numerical
problem well conditioned.

In [ ]:
def fingerprint(*parts):
    h = hashlib.sha256()
    for p in parts:
        h.update(np.asarray(p).tobytes() if isinstance(p, np.ndarray) else repr(p).encode())
    return h.hexdigest()[:12]

mom_keys = random.Random(SEED).sample(TRAIN, min(MOMENT_WINDOWS, len(TRAIN)))
mom_tasks = L.make_tasks("moments", mom_keys, CASES_PER_TASK, cache_dir=str(CACHE_DIR),
                         blocks=L.FULL_BLOCKS, stride=MOMENT_STRIDE)
t0 = time.time()
mom = L.run_parallel(mom_tasks, N_WORKERS, "standardiser")
MU, SD, CONST = L.moments_to_standardiser(mom)
NAMES = L.column_names()
print(f"standardiser from {len(mom_keys):,} windows in {time.time() - t0:.0f} s; "
      f"constant columns: {[NAMES[i] for i in np.where(CONST)[0]]}")
np.savez(OUT / "standardiser.npz", mu=MU, sd=SD, const=CONST, names=np.array(NAMES))

# LR4 has its own standardiser (same windows and rows, its wider design), so that everything
# above -- and every cached result of LR0-LR3 -- is computed exactly as before.
mom4_tasks = L.make_tasks("moments", mom_keys, CASES_PER_TASK, cache_dir=str(CACHE_DIR),
                          blocks=L.FULL_BLOCKS, lags=L.LAGS_LONG, stride=MOMENT_STRIDE)
t0 = time.time()
mom4 = L.run_parallel(mom4_tasks, N_WORKERS, "standardiser LR4", cache_dir=OUT / "_cache" /
                      f"mom4_{fingerprint(L.keys_sha(mom_keys), CASES_PER_TASK, MOMENT_STRIDE, L.LAGS_LONG)}")
MU4, SD4, CONST4 = L.moments_to_standardiser(mom4)
NAMES4 = L.column_names(L.FULL_BLOCKS, L.LAGS_LONG)
assert NAMES4[:len(NAMES)] == NAMES and len(NAMES4) == L.rung_cols("LR4_lagged40")
assert np.allclose(MU4[:len(MU)], MU, rtol=1e-9, atol=1e-9) and np.allclose(SD4[:len(SD)], SD, rtol=1e-9)
print(f"LR4 standardiser ({len(NAMES4)} columns) in {time.time() - t0:.0f} s; its first {len(MU)} "
      f"columns equal the one above")
np.savez(OUT / "standardiser_lr4.npz", mu=MU4, sd=SD4, const=CONST4, names=np.array(NAMES4))

## 5. Training passes

**Pass A** runs over every training window of the Stage-1 pool (978,260 windows) and every
sample. It builds the least-squares factor of the LR2 inputs. LR1's inputs are the leading
columns of LR2's, so the same factor also gives LR1 exactly.

**Pass B** builds LR3's factor (287 inputs) from a seeded 10 % of the pool, keeping every 5th
sample: about 2×10⁸ rows. The windows are split into two halves to check that the fit is stable.

**Pass C** builds LR4's factor from exactly the same windows, rows and halves as pass B, so LR3
and LR4 differ only in the memory span.

**Resuming.** Finished tasks are cached, so an interrupted run resumes where it stopped.

In [ ]:
P1, P2, P3 = (L.n_cols(L.RUNGS[r]) for r in ("LR1_pointwise", "LR2_line", "LR3_lagged"))
EXCL = L.excluded_inputs(NAMES)          # the two exact duplicates, see linear_regression_lib
print("excluded from every fit:", [NAMES[j] for j in EXCL])
# Cached tasks are named by their position, so the cache key includes the task size.
split_fp = fingerprint(L.keys_sha(TRAIN), MU, SD, CASES_PER_TASK)

tasks_a = L.make_tasks("fit", TRAIN, CASES_PER_TASK, cache_dir=str(CACHE_DIR),
                       blocks=L.RUNGS["LR2_line"], stride=1, mu=MU, sd=SD)
t0 = time.time()
res_a = L.run_parallel(tasks_a, N_WORKERS, "pass A", cache_dir=OUT / "_cache" / f"passA_{split_fp}")
FAC_A = L.combine_factors(res_a)
T_PASS_A = time.time() - t0
print(f"pass A: {sum(int(r['n_windows']) for r in res_a):,} windows, {FAC_A['n']:.3g} rows, "
      f"{T_PASS_A / 60:.1f} min")

if RUN_MODE == "smoke":
    # The worker pool must not change the fit: rerun pass A in this process and compare the
    # fitted MSE of both solutions on the same factor. (The pool's workers run single-threaded
    # LAPACK, this process multi-threaded, so the factors agree to rounding, not bit for bit.)
    t0 = time.time()
    fac_1 = L.combine_factors(L.run_parallel(tasks_a, 1, "pass A, one process"))
    sec_per_window = (time.time() - t0) / len(TRAIN)
    for p in (P1, P2):
        w_n, _ = L.ls_solve(FAC_A, p, exclude=EXCL)
        w_1, _ = L.ls_solve(fac_1, p, exclude=EXCL)
        for fac in (FAC_A, fac_1):
            a, b = L.fit_stats(w_n, fac)["mse"], L.fit_stats(w_1, fac)["mse"]
            assert abs(a - b) <= 1e-7 * b, (p, a, b)
    print(f"{N_WORKERS} workers and one process give the same fit (MSE to 1e-7) -- PASS")
    print(f"one process: {1e3 * sec_per_window:.1f} ms per training window -> pass A on the full "
          f"pool (978,260 windows) ~ {978260 * sec_per_window / N_WORKERS / 60:.0f} min "
          f"with {N_WORKERS} workers")

In [ ]:
b_keys = sorted(random.Random(SEED + 1).sample(TRAIN, max(2, round(LR3_WINDOW_FRACTION * len(TRAIN)))))
order = b_keys[:]
random.Random(SEED + 2).shuffle(order)
half = {k: i % 2 for i, k in enumerate(order)}
tasks_b = L.make_tasks("fit", b_keys, CASES_PER_TASK, cache_dir=str(CACHE_DIR),
                       blocks=L.RUNGS["LR3_lagged"], stride=LR3_STRIDE, mu=MU, sd=SD, n_groups=2)
for t in tasks_b:
    t["group_of"] = {(case, n_st, s): half[(t["lc"], case, n_st, s)]
                     for case, per_n in t["cases"] for n_st, ss in per_n for s in ss}
t0 = time.time()
res_b = L.run_parallel(tasks_b, N_WORKERS, "pass B",
                       cache_dir=OUT / "_cache" / f"passB_{fingerprint(split_fp, len(b_keys), LR3_STRIDE, L.LAGS)}")
FAC_B, FAC_B0, FAC_B1 = (L.combine_factors(res_b, g) for g in (None, 0, 1))
print(f"pass B: {len(b_keys):,} windows, {FAC_B['n']:.3g} rows, {(time.time() - t0) / 60:.1f} min")

In [ ]:
P4 = L.rung_cols("LR4_lagged40")
EXCL4 = L.excluded_inputs(NAMES4)
assert EXCL4 == EXCL                                # the duplicates are static inputs, ahead of any lag
tasks_c = L.make_tasks("fit", b_keys, LR4_CASES_PER_TASK, cache_dir=str(CACHE_DIR),
                       blocks=L.RUNGS["LR4_lagged40"], lags=L.LAGS_LONG, stride=LR4_STRIDE,
                       mu=MU4, sd=SD4, n_groups=2, budget_bytes=LR4_BUDGET, chunk_cap=LR4_CAP)
for t in tasks_c:
    t["group_of"] = {(case, n_st, s): half[(t["lc"], case, n_st, s)]
                     for case, per_n in t["cases"] for n_st, ss in per_n for s in ss}
t0 = time.time()
res_c = L.run_parallel(tasks_c, N_WORKERS, "pass C (LR4)", cache_dir=OUT / "_cache" /
                       f"passC_{fingerprint(L.keys_sha(b_keys), MU4, SD4, LR4_CASES_PER_TASK, LR4_STRIDE, L.LAGS_LONG)}")
FAC_C0, FAC_C1 = (L.combine_factors(res_c, g) for g in (0, 1))
FAC_C = L.merge_factors(FAC_C0, FAC_C1)             # the halves first: never re-stack every task
del res_c
assert FAC_C["n"] == FAC_B["n"], (FAC_C["n"], FAC_B["n"])   # the same rows as LR3
print(f"pass C: {len(b_keys):,} windows, {FAC_C['n']:.3g} rows, {(time.time() - t0) / 60:.1f} min")

## 6. Validation pass and the fits

**The validation factor** covers all 10,000 validation windows with every input and every
sample. It gives the exact validation MSE of any weights.

**The fits.**
- LR1 and LR2 are plain least squares.
- LR3 and LR4 take the ridge strength with the lowest validation MSE. If that is zero, they are
  plain least squares too. LR4's validation factor keeps every 5th sample (2×10⁷ rows).

**The memory curve.** LR4's columns are ordered LR3's first, then the longer lags in increasing
span, so one factor gives the fit for every span from ±5 s to the longest: the validation error
as a function of how much history the linear filter sees.

**GATE 7.** LR3 refitted from LR4's factor (its leading columns, the same rows) must reproduce
LR3 fitted from pass B.

**GATE 3.** The streamed least-squares path must reproduce an independent least-squares solve on
rows materialised from a few windows: scikit-learn's `LinearRegression`, or, where scikit-learn
is not installed, the same algorithm in numpy. The reference scales each input by its own
standard deviation first: the raw inputs span six orders of magnitude, and on LR4's 1,618 inputs
the solver's singular-value cutoff then drops genuine directions.

In [ ]:
tasks_v = L.make_tasks("fit", VAL, CASES_PER_TASK, cache_dir=str(CACHE_DIR),
                       blocks=L.FULL_BLOCKS, stride=1, mu=MU, sd=SD)
t0 = time.time()
res_v = L.run_parallel(tasks_v, N_WORKERS, "validation",
                       cache_dir=OUT / "_cache" / f"val_{fingerprint(L.keys_sha(VAL), MU, SD, CASES_PER_TASK)}")
FAC_V = L.combine_factors(res_v)
print(f"validation: {len(VAL):,} windows, {FAC_V['n']:.3g} rows, {(time.time() - t0) / 60:.1f} min")

tasks_v4 = L.make_tasks("fit", VAL, LR4_CASES_PER_TASK, cache_dir=str(CACHE_DIR), blocks=L.FULL_BLOCKS,
                        lags=L.LAGS_LONG, stride=LR4_VAL_STRIDE, mu=MU4, sd=SD4,
                        budget_bytes=LR4_BUDGET, chunk_cap=LR4_CAP)
t0 = time.time()
res_v4 = L.run_parallel(tasks_v4, N_WORKERS, "validation LR4", cache_dir=OUT / "_cache" /
                        f"val4_{fingerprint(L.keys_sha(VAL), MU4, SD4, LR4_CASES_PER_TASK, LR4_VAL_STRIDE, L.LAGS_LONG)}")
FAC_V4 = L.combine_factors(res_v4)
del res_v4
print(f"validation LR4: {len(VAL):,} windows, {FAC_V4['n']:.3g} rows, {(time.time() - t0) / 60:.1f} min")

WEIGHTS, FIT = {}, {}
FIT["LR0_static"] = dict(rank=None, train=L.fit_stats(L.static_weights(MU, SD, P2), FAC_A),
                         val=L.fit_stats(L.static_weights(MU, SD, P3), FAC_V))
for rung, p in (("LR1_pointwise", P1), ("LR2_line", P2)):
    w, info = L.ls_solve(FAC_A, p, exclude=EXCL)
    WEIGHTS[rung] = w
    FIT[rung] = dict(info, train=L.fit_stats(w, FAC_A), val=L.fit_stats(w, FAC_V))

ridge_scan = []
for lam in RIDGE_GRID:
    w, info = L.ls_solve(FAC_B, P3, ridge=lam, exclude=EXCL)
    ridge_scan.append((L.fit_stats(w, FAC_V)["mse"], lam, w, info))
best_mse, LAMBDA, w3, info3 = min(ridge_scan, key=lambda t: t[0])
WEIGHTS["LR3_lagged"] = w3
halves = [L.fit_stats(L.ls_solve(f, P3, ridge=LAMBDA, exclude=EXCL)[0], FAC_V)["mse"]
          for f in (FAC_B0, FAC_B1)]
FIT["LR3_lagged"] = dict(info3, train=L.fit_stats(w3, FAC_B), val=L.fit_stats(w3, FAC_V),
                         lambda_scan=[(lam, mse) for mse, lam, _w, _i in ridge_scan],
                         half_val_mse=halves,
                         half_rel_diff=abs(halves[0] - halves[1]) / min(halves))

ridge_scan4 = []
for lam in RIDGE_GRID:
    w, info = L.ls_solve(FAC_C, P4, ridge=lam, exclude=EXCL)
    ridge_scan4.append((L.fit_stats(w, FAC_V4)["mse"], lam, w, info))
best_mse4, LAMBDA4, w4, info4 = min(ridge_scan4, key=lambda t: t[0])
WEIGHTS["LR4_lagged40"] = w4
halves4 = [L.fit_stats(L.ls_solve(f, P4, ridge=LAMBDA4, exclude=EXCL)[0], FAC_V4)["mse"]
           for f in (FAC_C0, FAC_C1)]
w3_in4 = L.restandardise(w3, MU, SD, MU4, SD4)      # LR3's weights on LR4's standardised inputs
FIT["LR4_lagged40"] = dict(info4, train=L.fit_stats(w4, FAC_C), val=L.fit_stats(w4, FAC_V4),
                           lambda_scan=[(lam, mse) for mse, lam, _w, _i in ridge_scan4],
                           half_val_mse=halves4,
                           half_rel_diff=abs(halves4[0] - halves4[1]) / min(halves4),
                           lr3_on_same_val_rows=L.fit_stats(w3_in4, FAC_V4))

# GATE 7 -- LR3 is nested in LR4: refitted from pass C's factor (leading columns, same rows) it
# must reproduce LR3 fitted from pass B (column order, and both passes saw the same rows).
w3_c, _ = L.ls_solve(FAC_C, P3, ridge=LAMBDA, exclude=EXCL)
m_b, m_c = L.fit_stats(w3_in4, FAC_C)["mse"], L.fit_stats(w3_c, FAC_C)["mse"]
assert abs(m_b - m_c) <= 1e-6 * m_c, f"GATE 7 FAILED: LR3 from pass B {m_b} vs from pass C {m_c}"
print(f"GATE 7: LR3 refitted from LR4's factor reproduces LR3 (training MSE {m_c:.8g} vs "
      f"{m_b:.8g}) -- PASS")

# The memory curve: LR3 itself (12 lags, +-5 s), then the fit with every 1-s lag out to +-M, for
# every span M that LR4 holds (the first of those also fills LR3's gaps at +-3 and +-4 s).
M3 = max(abs(t) for t in L.LAGS)
MEMORY = []
for M in [M3] + sorted({abs(t) for t in L.LAGS_LONG if abs(t) > M3}):
    n_lags = len(L.LAGS) if M == M3 else max(i + 1 for i, t in enumerate(L.LAGS_LONG) if abs(t) <= M)
    assert all(abs(t) <= M for t in L.LAGS_LONG[:n_lags])          # a leading block of columns
    k = L.n_cols(L.FULL_BLOCKS, L.LAGS_LONG[:n_lags])
    w_m, _ = L.ls_solve(FAC_C, k, ridge=LAMBDA4, exclude=EXCL)
    st_m = L.fit_stats(w_m, FAC_V4)
    MEMORY.append(dict(span_s=M * 0.1, lags=n_lags, inputs=k, val_rmse=st_m["rmse"],
                       val_r2=st_m["r2"], train_rmse=L.fit_stats(w_m, FAC_C)["rmse"]))
FIT["LR4_lagged40"]["memory_curve"] = MEMORY
print("memory curve, validation RMSE [N] by span of the lags: " +
      ", ".join(f"+-{m['span_s']:g} s {m['val_rmse']:.1f}" for m in MEMORY))

print(f"{'model':15s} {'rank':>9s} {'train R2':>9s} {'val R2':>9s} {'val RMSE [N]':>13s}")
for rung, f in FIT.items():
    rk = "" if f.get("rank") is None else f"{f['rank']}/{f['size']}"
    print(f"{rung:15s} {rk:>9s} {f['train']['r2']:9.5f} {f['val']['r2']:9.5f} {f['val']['rmse']:13.1f}")
print(f"LR3 ridge lambda = {LAMBDA:g}; validation MSE of the two halves differ by "
      f"{100 * FIT['LR3_lagged']['half_rel_diff']:.3f} %")
print(f"LR4 ridge lambda = {LAMBDA4:g}; validation MSE of the two halves differ by "
      f"{100 * FIT['LR4_lagged40']['half_rel_diff']:.3f} %; on LR4's validation rows "
      f"(every {LR4_VAL_STRIDE}th sample) LR3 RMSE "
      f"{FIT['LR4_lagged40']['lr3_on_same_val_rows']['rmse']:.1f} N, "
      f"LR4 {FIT['LR4_lagged40']['val']['rmse']:.1f} N")

gate3_keys = random.Random(SEED + 3).sample(TRAIN, min(12, len(TRAIN)))
n_rows, err, ref_name = L.check_ols_vs_reference(CACHE_DIR, gate3_keys, MU, SD, L.RUNGS["LR2_line"],
                                                 exclude=EXCL)
assert err < 1e-6, f"GATE 3 FAILED: max |ours - {ref_name}| / std(y) = {err:.2e}"
print(f"GATE 3: {n_rows:,} rows, fitted values equal those of {ref_name} to {err:.1e} of std(y) -- PASS")
# LR4's rows are 1,618 wide, so its check uses three coarse-layout windows (N <= 6, 12,000-18,000
# rows: still ~10 rows per input, and ~1 GB in memory instead of ~5 GB at N = 21)
gate3_keys4 = random.Random(SEED + 4).sample([k for k in TRAIN if k[2] <= 6], 3)
n_rows, err, ref_name = L.check_ols_vs_reference(CACHE_DIR, gate3_keys4, MU4, SD4, L.FULL_BLOCKS,
                                                 exclude=EXCL, lags=L.LAGS_LONG)
assert err < 1e-6, f"GATE 3 (LR4) FAILED: max |ours - {ref_name}| / std(y) = {err:.2e}"
print(f"GATE 3 (LR4, {P4} inputs): {n_rows:,} rows, fitted values equal those of {ref_name} "
      f"to {err:.1e} of std(y) -- PASS")

In [ ]:
import csv
for rung, w in WEIGHTS.items():
    d = OUT / rung
    d.mkdir(parents=True, exist_ok=True)
    mu_r, sd_r, names_r = (MU4, SD4, NAMES4) if rung == "LR4_lagged40" else (MU, SD, NAMES)
    c0, c = L.raw_coefficients(w, mu_r, sd_r)
    with open(d / "coefficients.csv", "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["input", "coef_standardised", "coef_physical", "mean", "std"])
        wr.writerow(["intercept", w[0], c0, "", ""])
        for j in range(w.size - 1):
            wr.writerow([names_r[j], w[j + 1], c[j], mu_r[j], sd_r[j]])
np.savez(OUT / "weights.npz", **WEIGHTS)
with open(OUT / "fit_report.json", "w", encoding="utf-8") as f:
    json.dump(dict(mode=RUN_MODE, workers=N_WORKERS, cases_per_task=CASES_PER_TASK,
                   train_windows=len(TRAIN), val_windows=len(VAL),
                   lr3_windows=len(b_keys), lr3_stride=LR3_STRIDE, lags=list(L.LAGS),
                   lr4_lags=list(L.LAGS_LONG), lr4_stride=LR4_STRIDE, lr4_val_stride=LR4_VAL_STRIDE,
                   pass_a_minutes=T_PASS_A / 60, fits=FIT), f, indent=1, default=float)

top = np.argsort(-np.abs(WEIGHTS["LR1_pointwise"][1:]))[:10]
print("LR1: largest standardised weights ->",
      ", ".join(f"{NAMES[j]} {WEIGHTS['LR1_pointwise'][1 + j]:+.0f} N/sd" for j in top))

## 7. Test pass

**What is scored.** All five models on the 50,000 published test windows, with Stage 1's per-window metric
function. The results are aggregated with the port of Stage 1's `evaluate()`, twice:
- once over the published draw;
- once over the screened draw (the 49,513 windows the paper reports).

**Outputs.** The same files as Stage 1:
- `test_metrics.csv`, `test_metrics_by_node_count.csv` and `test_per_window_stats.csv`;
- a `test_timeseries_dump.npz` holding the per-window peaks and the snap events.

**Gates.**
- **GATE 4.** Everything that depends only on the ground truth must equal Stage 1's: the target
  sums, the flat-predictor term of the skill score, the MAPE gate and the tail thresholds.
- **GATE 5.** Each model's own per-window rows must rebuild its pooled metrics.
- **GATE 6.** LR0 is constant in time, so its skill score must be exactly 1 in every window.
- **GATE 8.** LR0-LR3 must reproduce the outputs of the first cluster run (job 872463): adding
  LR4 changes nothing for them.

In [ ]:
RUNG_ORDER = list(L.RUNGS)
tasks_t = L.make_tasks("eval", TEST, CASES_PER_TASK, cache_dir=str(CACHE_DIR),
                       weights={k: v for k, v in WEIGHTS.items()}, mu=MU, sd=SD,
                       stand={"LR4_lagged40": (MU4, SD4)}, rung_lags=dict(L.RUNG_LAGS),
                       rungs=RUNG_ORDER, snap_times=T["snap_times"], mape_min=cfg.tension_mape_min)
t0 = time.time()
res_t = L.run_parallel(tasks_t, N_WORKERS, "test")
RES = {}
for r in res_t:
    RES.update(r)
assert set(RES) == set(TEST)
print(f"test pass: {len(RES):,} windows x {len(RUNG_ORDER)} models in {(time.time() - t0) / 60:.1f} min")

SUMMARY = {}
for rung in RUNG_ORDER:
    acc_pub, acc_scr, dump_rows = L.EvalAccumulator(), L.EvalAccumulator(), []
    for key in TEST:                                   # the split's own order
        m, reg, drow = RES[key][rung]
        acc_pub.add(m, reg, key)
        if key in SCREENED:
            acc_scr.add(m, reg, key)
            dump_rows.append((key, drow))
    for acc, d in ((acc_pub, OUT / rung / "published"), (acc_scr, OUT / rung)):
        tm, per_nc = acc.finalize()
        L.write_test_outputs(d, tm, per_nc, acc.per_window_rows)
    L.write_dump(OUT / rung / "test_timeseries_dump.npz", dump_rows)
    SUMMARY[rung] = (acc_pub, acc_scr)
    print(f"[{rung}] wrote {OUT / rung}")

In [ ]:
import math

def close(a, b, rel):
    return abs(a - b) <= rel * max(1.0, abs(a), abs(b))

# GATE 4 -- the ground-truth side equals Stage 1's
TRUTH_FIELDS = ("n_pts", "y_sum", "y_sq", "td_mae_flat", "mape_valid_frac")
for name, rows, ref in (("published", SUMMARY["LR1_pointwise"][0].per_window_rows,
                         STAGE1_EVAL / "published" / "test_per_window_stats.csv"),
                        ("screened", SUMMARY["LR1_pointwise"][1].per_window_rows,
                         STAGE1_EVAL / "test_per_window_stats.csv")):
    if not ref.exists():
        assert not REQUIRE_REFERENCES, f"GATE 4 ({name}): reference {ref} not found"
        print(f"GATE 4 ({name}): reference not found -- skipped")
        continue
    ref_rows = {(int(r["lc_id"]), int(r["case_id"]), int(r["n_in"]), int(r["start_idx"])): r
                for r in L.read_rows(ref)}
    worst = 0.0
    for r in rows:
        q = ref_rows[(r["lc_id"], r["case_id"], r["n_in"], r["start_idx"])]
        for f in TRUTH_FIELDS:
            a, b = float(r[f]), float(q[f])
            worst = max(worst, abs(a - b) / max(1.0, abs(b)))
    assert worst < 1e-5, f"GATE 4 FAILED ({name}): worst relative difference {worst:.2e}"
    print(f"GATE 4 ({name}): {len(rows):,} windows, truth-side sums equal Stage 1's "
          f"(worst relative difference {worst:.1e}) -- PASS")
if RUN_MODE == "full" and not (STAGE1_EVAL / "test_metrics.csv").exists():
    assert not REQUIRE_REFERENCES, f"GATE 4 (tails): {STAGE1_EVAL / 'test_metrics.csv'} not found"
if RUN_MODE == "full" and (STAGE1_EVAL / "test_metrics.csv").exists():   # pooled over the whole draw
    ref = L.kv(STAGE1_EVAL / "test_metrics.csv")
    mine = L.kv(OUT / "LR1_pointwise" / "test_metrics.csv")
    for k in ("test_peak_tension_thr_p90", "test_peak_tension_thr_p99"):
        assert close(float(mine[k]), float(ref[k]), 1e-4), (k, mine[k], ref[k])
    print(f"GATE 4 (tails): thresholds T90 {float(mine['test_peak_tension_thr_p90']):.1f} N and "
          f"T99 {float(mine['test_peak_tension_thr_p99']):.1f} N equal Stage 1's -- PASS")

# GATE 5 -- per-window rows rebuild the pooled metrics
for rung in RUNG_ORDER:
    for acc, d in zip(SUMMARY[rung], (OUT / rung / "published", OUT / rung)):
        rows, tm = acc.per_window_rows, L.kv(d / "test_metrics.csv")
        n = sum(r["n_pts"] for r in rows)
        sst = sum(r["y_sq"] for r in rows) - sum(r["y_sum"] for r in rows) ** 2 / n
        r2 = 1.0 - sum(r["ss_res"] for r in rows) / sst
        mae = sum(r["mae"] for r in rows) / len(rows)
        skill = sum(r["td_skill"] for r in rows) / len(rows)
        assert close(r2, float(tm["test_global_R2_tension"]), 1e-9)
        assert close(mae, float(tm["test_MAE_tension"]), 1e-9)
        assert close(skill, float(tm["test_temporal_diff_skill_tension"]), 1e-9)
print("GATE 5: every model's per-window rows rebuild its pooled R2, MAE and skill -- PASS")

# GATE 6 -- a prediction constant in time has skill exactly 1
assert all(r["td_skill"] == 1.0 for r in SUMMARY["LR0_static"][0].per_window_rows)
print("GATE 6: LR0 skill score is exactly 1 in every window -- PASS")

# GATE 8 -- LR0-LR3 reproduce job 872463 (the run without LR4)
if RUN_MODE == "full":
    refs = [(REF_872463 / r / sub / "test_metrics.csv", OUT / r / sub / "test_metrics.csv")
            for r in ("LR0_static", "LR1_pointwise", "LR2_line", "LR3_lagged") for sub in ("", "published")]
    if all(a.exists() for a, _b in refs):
        worst = 0.0
        for ref_f, mine_f in refs:
            ref, mine = L.kv(ref_f), L.kv(mine_f)
            assert set(ref) == set(mine), (ref_f, set(ref) ^ set(mine))
            for k, v in ref.items():
                try:
                    a, b = float(mine[k]), float(v)
                except ValueError:                      # a non-numeric field, e.g. the empty test_loss
                    assert mine[k] == v, (ref_f, k, mine[k], v)
                    continue
                if math.isnan(a) and math.isnan(b):
                    continue
                worst = max(worst, abs(a - b) / max(1.0, abs(b)))
        assert worst < 1e-6, f"GATE 8 FAILED: LR0-LR3 differ from job 872463 by {worst:.2e}"
        print(f"GATE 8: LR0-LR3 reproduce job 872463 on both draws (worst relative difference "
              f"{worst:.1e}) -- PASS")
    else:
        assert not REQUIRE_REFERENCES, f"GATE 8: references under {REF_872463} not found"
        print(f"GATE 8: references under {REF_872463} not found -- skipped")

## 8. Stage 1 against the five linear models

**The draw.** Everything below is on the screened test draw, the one the paper reports.
Stage 1's row is recomputed from its own output files with the same functions, and it reproduces
the paper's tables.

**How to read the metrics.**
- **Pooled R² and MAE** are dominated by the static tension level, which differs strongly between
  stations and sites.
- **Station R² by region, the skill score S, the peaks and the snap events** measure whether a
  model follows the dynamic tension.
  - Station R² is computed per station over time.
  - S compares each prediction's step-to-step changes with the truth's; LOWER is better, and a
    constant-in-time prediction scores 1.

In [ ]:
import pandas as pd

MODEL_DIRS = {"Stage 1 (GAT+BiLSTM)": STAGE1_EVAL, **{r: OUT / r for r in RUNG_ORDER}}
MODEL_DIRS = {k: v for k, v in MODEL_DIRS.items() if (v / "test_metrics.csv").exists()}
if RUN_MODE == "smoke":
    MODEL_DIRS.pop("Stage 1 (GAT+BiLSTM)", None)   # different window set in smoke mode
comp = pd.DataFrame({name: L.summarise_run(d, T["snap_node"]) for name, d in MODEL_DIRS.items()})
LABELS = {
    "R2": "pooled R2", "MAE": "MAE [N]", "MAPE": "MAPE [%]", "S": "skill score S (lower = better)",
    "peak_MAE": "station-peak MAE [N]", "u90": "p90 tail: underprediction [%]",
    "undermass90": "p90 tail: under-mass [N]", "u99": "p99 tail: underprediction [%]",
    "undermass99": "p99 tail: under-mass [N]", "R2_grounded": "station R2, grounded (median)",
    "R2_touchdown": "station R2, touchdown (median)", "R2_suspended": "station R2, suspended (median)",
    "fl_ratio_dev": "fairlead peak, training sites: median pred/true",
    "fl_relerr_dev": "fairlead peak, training sites: mean |error| [%]",
    "fl_up_dev": "fairlead peak, training sites: underpredicted [%]",
    "fl_ratio_loc10": "fairlead peak, loc10: median pred/true",
    "fl_relerr_loc10": "fairlead peak, loc10: mean |error| [%]",
    "fl_ratio_loc11": "fairlead peak, loc11: median pred/true",
    "fl_relerr_loc11": "fairlead peak, loc11: mean |error| [%]",
    "nonsnap_u90": "p90 fairlead tail, no snap: underpredicted [%]",
    "nonsnap_bias90": "p90 fairlead tail, no snap: bias [N]",
    "snap_events": "snap events", "snap_capture_linemax": "snap capture, line maximum (median)",
    "snap_capture_own": "snap capture, own station (median)", "snap_own_n": "snap events at own station",
}
comp.index = [LABELS.get(i, i) for i in comp.index]
comp.to_csv(OUT / "lr_vs_stage1.csv")
pd.set_option("display.float_format", lambda v: f"{v:,.4g}")
comp

In [ ]:
rows = []
for name, d in MODEL_DIRS.items():
    for r in L.read_rows(d / "test_metrics_by_node_count.csv"):
        rows.append(dict(model=name, N=int(r["node_count"]), R2=float(r["global_R2_tension"]),
                         MAE=float(r["MAE_tension"]), MAPE=float(r["MAPE_tension"]),
                         S=float(r["temporal_diff_skill_tension"]),
                         peak_MAE=float(r["peak_tension_MAE"])))
by_layout = pd.DataFrame(rows)
by_layout.to_csv(OUT / "lr_vs_stage1_by_layout.csv", index=False)
by_layout.pivot(index="N", columns="model", values=["R2", "MAE", "S"])

## 9. Figures

In [ ]:
import matplotlib.pyplot as plt

FIG = OUT / "figures"
FIG.mkdir(exist_ok=True)
names = list(MODEL_DIRS)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (col, lab) in zip(axes, (("R2", "pooled R$^2$"), ("MAE", "MAE [N]"), ("S", "skill score S"))):
    for name in names:
        d = by_layout[by_layout.model == name].sort_values("N")
        ax.plot(d.N, d[col], marker="o", ms=3, label=name)
    ax.set_xlabel("stations N"); ax.set_ylabel(lab); ax.grid(alpha=0.3)
    if col == "MAE":
        ax.set_yscale("log")
axes[0].legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIG / "metrics_by_layout.png", dpi=180); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(names), figsize=(3.1 * len(names), 3.2), sharex=True, sharey=True)
for ax, name in zip(np.atleast_1d(axes), names):
    z = np.load(MODEL_DIRS[name] / "test_timeseries_dump.npz")
    t, p = z["peak_true_all"] / 1e3, z["peak_pred_all"] / 1e3
    ax.scatter(t, p, s=2, alpha=0.3, lw=0)
    lim = [0, max(t.max(), p.max()) * 1.03]
    ax.plot(lim, lim, "k--", lw=0.8)
    ax.set_title(name, fontsize=8); ax.set_xlabel("true fairlead peak [kN]")
np.atleast_1d(axes)[0].set_ylabel("predicted [kN]")
fig.tight_layout(); fig.savefig(FIG / "fairlead_peak_parity.png", dpi=180); plt.show()

**The memory curve.** LR4's validation error as a function of the span of its lags, from LR3's
±5 s to ±40 s (fitted from one factor, section 6). If more history helped a linear model, the
curve would keep falling.

In [ ]:
mem = FIT["LR4_lagged40"]["memory_curve"]
fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot([m["span_s"] for m in mem], [m["val_rmse"] for m in mem], marker="o", ms=3, color="C4",
        label="linear filter, validation")
ax.plot([m["span_s"] for m in mem], [m["train_rmse"] for m in mem], ls="--", color="C4", lw=0.8,
        label="linear filter, training")
ax.plot([mem[0]["span_s"]], [mem[0]["val_rmse"]], "s", color="C3", ms=6, label="LR3 (12 lags)")
ax.set_xlabel("span of the lags, \u00b1 [s]"); ax.set_ylabel("RMSE [N]"); ax.grid(alpha=0.3)
ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIG / "lr4_memory_curve.png", dpi=180); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.4))
bins = np.linspace(0, 1.6, 81)
for name in names:
    s = np.array([float(r["td_skill"]) for r in L.read_rows(MODEL_DIRS[name] / "test_per_window_stats.csv")])
    ax.hist(np.clip(s, bins[0], bins[-1]), bins=bins, histtype="step", density=True, label=name)
ax.set_xlabel("per-window skill score S (lower = better; 1 = no better than constant)")
ax.set_ylabel("density"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIG / "skill_histogram.png", dpi=180); plt.show()

**Time series.** These are the four windows of the paper's along-line figure: layouts 6, 10, 15
and 21, with both withheld sites among them. Each row shows three stations:
- a grounded station;
- the touchdown station shown in the paper;
- the fairlead.

Each panel draws five curves: the truth, Stage 1, and the linear models LR1, LR3 and LR4.

In [ ]:
ALONG_ROWS = [(6, "v5", "ty2", 3), (10, "v5", "nc10_sn0", 5), (15, "snapfix", "nc15_ty1", 11),
              (21, "snapfix", "nc21_ty1", 11)]
have = all(p.exists() for p in STAGE1_DUMPS.values())
assert have or not REQUIRE_REFERENCES, f"Stage-1 dumps not found: {list(STAGE1_DUMPS.values())}"
if have:
    dumps = {k: np.load(v) for k, v in STAGE1_DUMPS.items()}
    metas = [tuple(int(x) for x in dumps[src][f"{key}_meta"]) for _n, src, key, _st in ALONG_ROWS]
    lr = L.predict_windows(CACHE_DIR, metas,
                           {k: WEIGHTS[k] for k in ("LR1_pointwise", "LR3_lagged", "LR4_lagged40")},
                           MU, SD, lags=L.LAGS_LONG, stand={"LR4_lagged40": (MU4, SD4)})
    tt = np.arange(L.WINDOW_LEN) * 0.1
    fig, axes = plt.subplots(len(ALONG_ROWS), 3, figsize=(13, 2.3 * len(ALONG_ROWS)), sharex=True)
    for r, ((n_st, src, key, td), meta) in enumerate(zip(ALONG_ROWS, metas)):
        s1_true, s1_pred = dumps[src][f"{key}_true"], dumps[src][f"{key}_pred"]
        mine = lr[meta]
        assert np.allclose(mine["true"], s1_true, rtol=1e-5, atol=1e-2)   # same window, same truth
        for c, (st, lab) in enumerate(((1, "grounded"), (td, "touchdown"), (n_st - 1, "fairlead"))):
            ax = axes[r, c]
            ax.plot(tt, s1_true[:, st] / 1e3, "k", lw=1.0, label="truth")
            ax.plot(tt, s1_pred[:, st] / 1e3, color="C0", lw=0.8, label="Stage 1")
            ax.plot(tt, mine["LR1_pointwise"][:, st] / 1e3, color="C1", lw=0.8, label="LR1")
            ax.plot(tt, mine["LR3_lagged"][:, st] / 1e3, color="C3", lw=0.8, label="LR3")
            ax.plot(tt, mine["LR4_lagged40"][:, st] / 1e3, color="C4", lw=0.8, label="LR4")
            ax.set_title(f"loc{meta[0]:02d}/{meta[1]:04d}, N={n_st}, station {st} ({lab})", fontsize=8)
            if c == 0:
                ax.set_ylabel("T [kN]")
    axes[0, 0].legend(fontsize=7, ncol=5)
    for ax in axes[-1]:
        ax.set_xlabel("t [s]")
    fig.tight_layout(); fig.savefig(FIG / "timeseries_along_line_windows.png", dpi=180); plt.show()
    np.savez(FIG / "timeseries_along_line_windows.npz",
             **{f"row{r}_{k}": v for r, m in enumerate(metas) for k, v in lr[m].items()})
else:
    print("Stage-1 dumps not found -- time-series figure skipped")

**The input–output relation itself (ground truth only).**
- A predicted-against-true parity plot is a straight line for ANY accurate model, so it cannot show
  whether the map from motion to tension is linear.
- This plot shows the map directly: the fairlead tension against the fairlead's horizontal offset,
  for one simulation per site, after the 30 s start-up ramp.
- A linear model would need one straight line shared by all sites.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for i, lc in enumerate(L.LOC_IDS):
    case = next(c for c in range(10, 301) if (lc, c) not in L.CORRUPT_CASES and (lc, c) not in T["eval_mask"])
    arr = L.load_case(CACHE_DIR, lc, case)
    x, ten = arr["x"][300:, -1], arr["T"][300:, -1]
    ax.scatter(x - arr["x"][0, -1], ten / 1e3, s=1, alpha=0.25, lw=0, color=f"C{i}",
               label=f"loc{lc:02d}/{case:04d}")
ax.set_xlabel("fairlead horizontal offset from t = 0 [m]"); ax.set_ylabel("fairlead tension [kN]")
ax.legend(fontsize=6, markerscale=8, ncol=2); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIG / "fairlead_tension_vs_offset.png", dpi=180); plt.show()